In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
RAW_DATA_DIR = "../data/raw"


books = pd.read_csv(f"{RAW_DATA_DIR}/Books.csv", low_memory=False)
ratings = pd.read_csv(f"{RAW_DATA_DIR}/Ratings.csv", low_memory=False)
users = pd.read_csv(f"{RAW_DATA_DIR}/Users.csv", low_memory=False)

In [ ]:
books

In [ ]:
ratings

In [ ]:
users

In [ ]:
# Check for null values in each DataFrame
for name, df in [("Books", books), ("Ratings", ratings), ("Users", users)]:
    print(f"\n{name}:")
    null_counts = df.isnull().sum()
    for column, count in null_counts.items():
        if count > 0:
            print(f"\033[91m {column:<25}: {count} null values\033[0m")
        else:
            print(f" {column:<25}: No null values")

In [ ]:
# Check for duplicate entries in each DataFrame
for name, df in [("Books", books), ("Ratings", ratings), ("Users", users)]:
    print(f"\n{name}:")
    duplicate_count = df.duplicated().sum()
    if duplicate_count == 0:
        print(" No duplicate entries found.")
    else:
        print(f" {duplicate_count:,} duplicate entries found.")

In [ ]:
# Check rating distribution
rating_counts = ratings["Book-Rating"].value_counts().sort_index()
for rating, count in rating_counts.items():
    print(f" {rating}: {count:,} ratings ({count / len(ratings) * 100:.1f}%)")

In [ ]:
# Plot rating distribution
plt.figure(figsize=(8, 5))
plt.bar(rating_counts.index, rating_counts.values, color="skyblue")
plt.xlabel("Book Rating")
plt.ylabel("Number of Ratings")
plt.title("Distribution of Book Ratings")
plt.xticks(rating_counts.index)
plt.grid(axis="y", alpha=0.75)
plt.show()

In [ ]:
# Check how many unque users and books are in the ratings dataset
unique_users = ratings["User-ID"].nunique()
unique_books = ratings["ISBN"].nunique()
print(f"Unique users in ratings: {unique_users:,}")
print(f"Unique books in ratings: {unique_books:,}")
print(f"Non-unique users: {len(ratings) - unique_users:,}")
print(f"Non-unique books: {len(ratings) - unique_books:,}")

In [ ]:
# Check how many books have few ratings (0, 1, 2) 
book_rating_counts = ratings["ISBN"].value_counts()
books_with_no_ratings = book_rating_counts[book_rating_counts == 0].count()
books_with_one_rating = book_rating_counts[book_rating_counts == 1].count()
books_with_two_ratings = book_rating_counts[book_rating_counts == 2].count()
print(f"Books with no ratings: {books_with_no_ratings:,} ({books_with_no_ratings / unique_books * 100:.1f}%)")
print(f"Books with one rating: {books_with_one_rating:,} ({books_with_one_rating / unique_books * 100:.1f}%)")
print(f"Books with two ratings: {books_with_two_ratings:,} ({books_with_two_ratings / unique_books * 100:.1f}%)")
print(f"Books with 0-2 ratings: {books_with_no_ratings + books_with_one_rating + books_with_two_ratings:,} ({(books_with_no_ratings + books_with_one_rating + books_with_two_ratings) / unique_books * 100:.1f}%)")

# Check how many users have only rated one book
user_rating_counts = ratings["User-ID"].value_counts()
users_with_one_rating = user_rating_counts[user_rating_counts == 1].count()
print(f"Users with only one rating: {users_with_one_rating:,} ({users_with_one_rating / unique_users * 100:.1f}%)")

In [ ]:
# Sort books by rating count and show top 10 (excluding those with zero ratings)
top_books = book_rating_counts.sort_values(ascending=False).head(10)
print("\nTop 10 most rated books:")

for isbn, count in top_books.items():
    matching = books[books["ISBN"] == isbn]["Book-Title"]
    if not matching.empty:
        book_title = matching.iloc[0]
        print(f" {book_title} (ISBN: {isbn}): {count:,} ratings")
    else:
        print(f" [UNKNOWN TITLE] (ISBN: {isbn}): {count:,} ratings")

# Show bottom 10 least rated books (excluding those with zero ratings)
bottom_books = book_rating_counts.sort_values(ascending=True).head(10)
print("\nBottom 10 least rated books:")
for isbn, count in bottom_books.items():
    matching = books[books["ISBN"] == isbn]["Book-Title"]
    if not matching.empty:
        book_title = matching.iloc[0]
        print(f" {book_title} (ISBN: {isbn}): {count:,} ratings")
    else:
        print(f" [UNKNOWN TITLE] (ISBN: {isbn}): {count:,} ratings")

In [ ]:
# Books by average rating (only include books with at least 10 ratings)
book_avg_ratings = ratings.groupby("ISBN")["Book-Rating"].mean()
book_rating_counts = ratings["ISBN"].value_counts()
books_with_10_or_more_ratings = book_rating_counts[book_rating_counts >= 10].index
avg_ratings_for_popular_books = book_avg_ratings[books_with_10_or_more_ratings].sort_values(ascending=False)
print("\nTop 10 highest rated books (with at least 10 ratings):")
for isbn, avg_rating in avg_ratings_for_popular_books.head(10).items():
    matching = books[books["ISBN"] == isbn]["Book-Title"]
    if not matching.empty:
        book_title = matching.iloc[0]
        print(f" {book_title} (ISBN: {isbn}): Average Rating {avg_rating:.2f}")
    else:
        print(f" [UNKNOWN TITLE] (ISBN: {isbn}): Average Rating {avg_rating:.2f}")

In [ ]:
# How many books survive data cleaning
# Exclude ratings = 0, remove duplicate user-ISBN pairs
ratings_clean = ratings[ratings["Book-Rating"] > 0]
ratings_clean = ratings_clean.drop_duplicates(["User-ID", "ISBN"])

# Exclude books with less than 3 ratings
book_counts = ratings_clean["ISBN"].value_counts()
popular_isbns = book_counts[book_counts >= 3].index

# Exclude missing titles/authors and duplicate ISBN entries
books_clean = books.dropna(subset=["Book-Title", "Book-Author"])
books_clean = books_clean.drop_duplicates("ISBN")

# Calculate how many books survive cleaning
books_final = books_clean[books_clean["ISBN"].isin(popular_isbns)]

total_raw      = books["ISBN"].nunique()
total_popular  = len(popular_isbns)
total_survived = len(books_final)
orphans        = total_popular - total_survived

print(f"Total Unique ISBNs (Raw)          : {total_raw:,}")
print(f"Books with >= 3 explicit ratings  : {total_popular:,}")
print(f"ISBNs in ratings but not in books : {orphans:,}")
print(f"Books that survive cleaning       : {total_survived:,}")